In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

In [58]:
df = pd.read_csv('../dataset_final_para_ia.csv')
df = df[df['estado'] != 'deforestacion']
df = df[df['estado'] != 'enfermo_sequia']
df = df.sort_values(by='latitud').reset_index(drop=True) 
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
# 2. Target
mapeo = dict(enumerate(df['estado'].astype('category').cat.categories))
print(mapeo)
Y = df['estado'].astype('category').cat.codes
X = df.drop(columns=['estado', 'latitud', 'longitud', 'fecha_captura_real'], errors='ignore')
X = X.select_dtypes(include=[np.number])

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = X_train.replace([np.inf, -np.inf], np.nan)

# 2. Ahora que NO hay infinitos, calculamos las medias reales y limpias
medias_limpias = X_train.mean()

# 3. Rellenamos los NaN con esas medias correctas
X_train = X_train.fillna(medias_limpias)

X_train_scaled = scaler.fit_transform(X_train)

{0: 'enfermo_plaga', 1: 'incendio', 2: 'sano'}


In [59]:
class RedNeuronal(nn.Module):
    def __init__(self):
        super(RedNeuronal, self).__init__()
        self.fc1 = nn.Linear(in_features=X_train_scaled.shape[1], out_features=64)
        self.dropout1 = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(in_features=64, out_features=32)
        self.dropout2 = nn.Dropout(p=0.3)
        self.fc3 = nn.Linear(in_features=32, out_features=len(mapeo))

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)  # Capa de salida final
        return x

In [65]:
modelo = RedNeuronal()

pesos = compute_class_weight(class_weight='balanced', 
                             classes=np.unique(Y_train), 
                             y=Y_train)

# 2. Pasamos esos pesos a PyTorch
pesos_tensor = torch.tensor(pesos, dtype=torch.float32)

# 3. Y le damos los pesos a nuestra función de pérdida
criterion = nn.CrossEntropyLoss(weight=pesos_tensor)
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.001)

for epoch in range(850):
    modelo.train()
    inputs = torch.tensor(X_train_scaled, dtype=torch.float32)
    labels = torch.tensor(Y_train.values, dtype=torch.long)
    
    optimizer.zero_grad()
    outputs = modelo(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        _, predicciones = torch.max(outputs, 1)
        f1 = f1_score(labels.cpu().numpy(), predicciones.cpu().numpy(), average='macro')
        
    print(f"Época {epoch+1} -> Loss: {loss.item():.4f} | F1-Score: {f1:.4f}")

with torch.no_grad(): # Desactivamos gradientes para calcular métricas
        # El modelo devuelve probabilidades/logits por clase. 
        # Tomamos el índice de la clase con el valor más alto (la predicción final).
        _, predicciones = torch.max(outputs, 1)
        
        # Pasamos los tensores a arrays de NumPy
        y_true = labels.cpu().numpy()
        y_pred = predicciones.cpu().numpy()
        
        # Calculamos el F1-Score. 
        # Usamos 'macro' o 'weighted' si tienes más de 2 clases (multiclase)
        f1 = f1_score(y_true, y_pred, average='macro')
        
        print(f"Época {epoch+1} -> Loss: {loss.item():.4f} | F1-Score: {f1:.4f}")

Época 1 -> Loss: 1.1007 | F1-Score: 0.2738
Época 2 -> Loss: 1.0901 | F1-Score: 0.3052
Época 3 -> Loss: 1.0834 | F1-Score: 0.3408
Época 4 -> Loss: 1.0769 | F1-Score: 0.3675
Época 5 -> Loss: 1.0696 | F1-Score: 0.4106
Época 6 -> Loss: 1.0623 | F1-Score: 0.4159
Época 7 -> Loss: 1.0534 | F1-Score: 0.4542
Época 8 -> Loss: 1.0455 | F1-Score: 0.4603
Época 9 -> Loss: 1.0369 | F1-Score: 0.4935
Época 10 -> Loss: 1.0296 | F1-Score: 0.4997
Época 11 -> Loss: 1.0216 | F1-Score: 0.5145
Época 12 -> Loss: 1.0146 | F1-Score: 0.5200
Época 13 -> Loss: 1.0067 | F1-Score: 0.5295
Época 14 -> Loss: 0.9986 | F1-Score: 0.5345
Época 15 -> Loss: 0.9917 | F1-Score: 0.5425
Época 16 -> Loss: 0.9836 | F1-Score: 0.5508
Época 17 -> Loss: 0.9750 | F1-Score: 0.5607
Época 18 -> Loss: 0.9659 | F1-Score: 0.5620
Época 19 -> Loss: 0.9524 | F1-Score: 0.5788
Época 20 -> Loss: 0.9477 | F1-Score: 0.5843
Época 21 -> Loss: 0.9392 | F1-Score: 0.5816
Época 22 -> Loss: 0.9286 | F1-Score: 0.5914
Época 23 -> Loss: 0.9169 | F1-Score: 0.60

In [66]:
# 1. Ponemos el modelo en modo evaluación (apaga Dropout, Batchnorm, etc.)
modelo.eval()

# 2. Escalar X_test usando el MISMO scaler que usamos en train. 
# OJO: Usamos transform(), NO fit_transform(), para no romper la escala.
X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_test = X_test.fillna(medias_limpias) # Usamos las medias que calculamos en train
X_test_scaled = scaler.transform(X_test)

# 3. Convertir a tensores
inputs_test = torch.tensor(X_test_scaled, dtype=torch.float32)
labels_test = torch.tensor(Y_test.values, dtype=torch.long)

# 4. Hacer las predicciones SIN calcular gradientes (ahorra memoria y tiempo)
with torch.no_grad():
    outputs_test = modelo(inputs_test)
    _, predicciones_test = torch.max(outputs_test, 1)

# 5. Pasamos los resultados a NumPy para verlos con Scikit-Learn
y_real = labels_test.cpu().numpy()
y_pred = predicciones_test.cpu().numpy()

# 6. Mostrar el reporte final
print("--- REPORTE DE EVALUACIÓN EN DATOS NUEVOS (TEST) ---")
print(classification_report(y_real, y_pred, target_names=mapeo.values()))

--- REPORTE DE EVALUACIÓN EN DATOS NUEVOS (TEST) ---
               precision    recall  f1-score   support

enfermo_plaga       0.94      0.91      0.93       434
     incendio       0.79      0.82      0.80       247
         sano       0.86      0.86      0.86       387

     accuracy                           0.87      1068
    macro avg       0.86      0.86      0.86      1068
 weighted avg       0.87      0.87      0.87      1068

